# TransMorph Registration & Folding Correction

End-to-end example: train a small
[TransMorph](https://github.com/junyuchen245/TransMorph_Pytorch)
registration network on synthetic 2D images and correct folding with
DVFopt.

TransMorph uses a Swin-Transformer encoder + ConvNet decoder to predict
displacement fields.  Compared to VoxelMorph's pure CNN, the
self-attention mechanism captures longer-range spatial relationships but
can produce more folding when trained without diffeomorphic integration.

We train two variants:
1. **TransMorph (direct)** — `integration_steps=0`, more folding
2. **TransMorph (diffeomorphic)** — `integration_steps=7`, less folding

**Requirements:**
```
pip install timm
```
Plus PyTorch — see `setup-venv.bat`.

In [ ]:
import time
import importlib

import numpy as np
import matplotlib.pyplot as plt

from dvfopt import jacobian_det2D, iterative_parallel
from dvfopt.viz import (
    plot_grid_before_after,
    plot_checkerboard_before_after,
    plot_neg_jdet_neighborhoods,
)

HAS_TORCH = importlib.util.find_spec("torch") is not None
HAS_TIMM = importlib.util.find_spec("timm") is not None
print(f"PyTorch: {'available' if HAS_TORCH else 'MISSING'}")
print(f"timm   : {'available' if HAS_TIMM else 'MISSING — pip install timm'}")

## Configuration

In [ ]:
IMAGE_SIZE = 64           # HxW for synthetic images (keep small for training speed)
NUM_TRAIN_IMAGES = 200
NUM_TEST_PAIRS = 4

# Training
EPOCHS = 200
STEPS_PER_EPOCH = 50
LR = 1e-3
LAMBDA_SMOOTH = 0.05

# Correction
JDET_THRESHOLD = 0.01

DEVICE = None  # auto-detect below

## Generate synthetic dataset

In [ ]:
def make_random_image(size, rng):
    img = np.zeros((size, size), dtype=np.float32)
    yy, xx = np.mgrid[0:size, 0:size].astype(np.float32)
    n_shapes = rng.integers(1, 5)
    for _ in range(n_shapes):
        cy = rng.uniform(size * 0.15, size * 0.85)
        cx = rng.uniform(size * 0.15, size * 0.85)
        ry = rng.uniform(size * 0.06, size * 0.25)
        rx = rng.uniform(size * 0.06, size * 0.25)
        intensity = rng.uniform(0.3, 1.0)
        mask = ((yy - cy) / ry) ** 2 + ((xx - cx) / rx) ** 2 <= 1.0
        img[mask] = intensity
    return img


rng = np.random.default_rng(42)
n_total = NUM_TRAIN_IMAGES + NUM_TEST_PAIRS * 2
images = np.stack([make_random_image(IMAGE_SIZE, rng) for _ in range(n_total)])
train_images = images[:NUM_TRAIN_IMAGES]
test_images = images[NUM_TRAIN_IMAGES:]

print(f"Training: {train_images.shape}  Test: {test_images.shape}")

fig, axes = plt.subplots(1, 6, figsize=(14, 2.5))
for i, ax in enumerate(axes):
    ax.imshow(train_images[i], cmap="gray", vmin=0, vmax=1)
    ax.set_title(f"#{i}"); ax.axis("off")
plt.suptitle("Sample training images")
plt.tight_layout()
plt.show()

## Build TransMorph-style model

We build a lightweight transformer encoder + CNN decoder that mirrors
the TransMorph architecture using `timm` for the Swin Transformer
backbone and VoxelMorph's spatial transformer for warping.

In [ ]:
if HAS_TORCH and HAS_TIMM:
    import torch
    import torch.nn as nn
    import torch.nn.functional as F
    import timm

    DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
    print(f"Device: {DEVICE}")

    class SwinRegNet(nn.Module):
        """Lightweight Swin-Transformer registration network for 2D images.

        Uses a pretrained Swin-Tiny backbone (adapted for 2-channel input:
        concatenated source + target) followed by a small ConvNet decoder
        that outputs a 2-channel displacement field.
        """

        def __init__(self, img_size=64, integration_steps=0):
            super().__init__()
            self.integration_steps = integration_steps

            # Encoder: Swin-Tiny with 2-channel input
            self.encoder = timm.create_model(
                "swin_tiny_patch4_window7_224",
                pretrained=False,
                in_chans=2,
                num_classes=0,
                global_pool="",
                img_size=img_size,
            )

            # Get encoder output dim
            with torch.no_grad():
                dummy = torch.zeros(1, 2, img_size, img_size)
                enc_out = self.encoder(dummy)
                if enc_out.ndim == 3:
                    # (B, N, C) — reshape to spatial
                    B, N, C = enc_out.shape
                    h = w = int(N ** 0.5)
                    self.enc_spatial = (h, w)
                    self.enc_channels = C
                else:
                    self.enc_spatial = enc_out.shape[2:]
                    self.enc_channels = enc_out.shape[1]

            # Decoder: upsample back to full resolution
            self.decoder = nn.Sequential(
                nn.Conv2d(self.enc_channels, 64, 3, padding=1),
                nn.ReLU(inplace=True),
                nn.Upsample(size=img_size, mode="bilinear", align_corners=False),
                nn.Conv2d(64, 32, 3, padding=1),
                nn.ReLU(inplace=True),
                nn.Conv2d(32, 2, 3, padding=1),  # (B, 2, H, W) displacement
            )

            # Small init for displacement head
            nn.init.normal_(self.decoder[-1].weight, std=1e-5)
            nn.init.zeros_(self.decoder[-1].bias)

            self.img_size = img_size

        def _integrate(self, velocity):
            """Scaling-and-squaring integration for diffeomorphic fields."""
            disp = velocity / (2 ** self.integration_steps)
            for _ in range(self.integration_steps):
                disp = disp + self._warp(disp, disp)
            return disp

        def _warp(self, src, flow):
            """Warp src by flow using grid_sample."""
            B, C, H, W = src.shape
            grid_y, grid_x = torch.meshgrid(
                torch.linspace(-1, 1, H, device=src.device),
                torch.linspace(-1, 1, W, device=src.device),
                indexing="ij",
            )
            grid = torch.stack([grid_x, grid_y], dim=-1).unsqueeze(0).expand(B, -1, -1, -1)
            # Normalize flow to [-1, 1]
            flow_norm = torch.stack([
                flow[:, 1] / (W / 2),  # dx
                flow[:, 0] / (H / 2),  # dy
            ], dim=-1)
            return F.grid_sample(
                src, grid + flow_norm,
                mode="bilinear", padding_mode="border", align_corners=False
            )

        def forward(self, source, target):
            """Returns (displacement, warped_source)."""
            x = torch.cat([source, target], dim=1)  # (B, 2, H, W)
            enc = self.encoder(x)

            if enc.ndim == 3:
                B, N, C = enc.shape
                h, w = self.enc_spatial
                enc = enc.permute(0, 2, 1).reshape(B, C, h, w)

            velocity = self.decoder(enc)  # (B, 2, H, W)

            if self.integration_steps > 0:
                disp = self._integrate(velocity)
            else:
                disp = velocity

            warped = self._warp(source, disp)
            return disp, warped

    print("SwinRegNet model defined")
else:
    print("PyTorch or timm not available — cannot define model")

## Train & evaluate both variants

Train each model, register the test pairs, and check for folding.

In [ ]:
def train_and_register(integration_steps, label):
    """Train a SwinRegNet and return test displacements."""
    print(f"\n{'='*60}")
    print(f"  {label}  (integration_steps={integration_steps})")
    print(f"{'='*60}")

    model = SwinRegNet(img_size=IMAGE_SIZE, integration_steps=integration_steps).to(DEVICE)
    n_params = sum(p.numel() for p in model.parameters())
    print(f"  Parameters: {n_params:,}")

    optimizer = torch.optim.Adam(model.parameters(), lr=LR)
    train_t = torch.from_numpy(train_images).float().unsqueeze(1).to(DEVICE)
    n_train = train_t.shape[0]

    model.train()
    for epoch in range(EPOCHS):
        epoch_loss = 0.0
        for _ in range(STEPS_PER_EPOCH):
            idx = torch.randint(0, n_train, (2,))
            source = train_t[idx[0]].unsqueeze(0)
            target = train_t[idx[1]].unsqueeze(0)

            optimizer.zero_grad()
            disp, warped = model(source, target)

            # MSE image loss
            img_loss = F.mse_loss(warped, target)
            # Smoothness: spatial gradient L2
            dy = torch.abs(disp[:, :, 1:, :] - disp[:, :, :-1, :])
            dx = torch.abs(disp[:, :, :, 1:] - disp[:, :, :, :-1])
            smooth_loss = (dy.mean() + dx.mean()) / 2

            loss = img_loss + LAMBDA_SMOOTH * smooth_loss
            loss.backward()
            optimizer.step()
            epoch_loss += loss.item()

        if (epoch + 1) % 50 == 0 or epoch == 0:
            print(f"  Epoch {epoch+1:>4d}/{EPOCHS}  loss={epoch_loss/STEPS_PER_EPOCH:.6f}")

    # Register test pairs
    model.eval()
    test_t = torch.from_numpy(test_images).float().unsqueeze(1).to(DEVICE)
    test_results = []

    with torch.no_grad():
        for i in range(NUM_TEST_PAIRS):
            source = test_t[2 * i].unsqueeze(0)
            target = test_t[2 * i + 1].unsqueeze(0)

            t0 = time.perf_counter()
            disp, warped = model(source, target)
            inf_time = time.perf_counter() - t0

            disp_np = disp.cpu().numpy()[0]  # (2, H, W)
            H, W = disp_np.shape[1], disp_np.shape[2]
            deformation = np.zeros((3, 1, H, W), dtype=np.float64)
            deformation[1, 0] = disp_np[0]  # dy
            deformation[2, 0] = disp_np[1]  # dx

            test_results.append({
                "deformation": deformation,
                "inf_time": inf_time,
                "source": source.cpu().numpy()[0, 0],
                "target": target.cpu().numpy()[0, 0],
                "warped": warped.cpu().numpy()[0, 0],
            })

    return test_results

In [ ]:
all_variant_results = {}

if HAS_TORCH and HAS_TIMM:
    for steps, label in [(0, "TransMorph (direct)"), (7, "TransMorph (diffeo)")]:
        all_variant_results[label] = train_and_register(steps, label)
else:
    print("PyTorch or timm not available — skipping training")

## Check Jacobians & correct folding

In [ ]:
def to_dvfopt_and_correct(deformation, label):
    phi_init = np.stack([deformation[1, 0], deformation[2, 0]])
    jac = jacobian_det2D(phi_init)
    n_neg = int((jac <= 0).sum())
    H, W = deformation.shape[-2:]

    print(f"  {label:<35s}  {H}x{W}  neg={n_neg:>4d}  min_jdet={jac.min():+.6f}", end="")

    if n_neg == 0:
        print("  [no correction needed]")
        return None

    t0 = time.perf_counter()
    phi = iterative_parallel(deformation.copy(), verbose=0, threshold=JDET_THRESHOLD)
    corr_time = time.perf_counter() - t0

    jac_final = jacobian_det2D(phi)
    n_final = int((jac_final <= 0).sum())
    l2 = float(np.sqrt(np.sum((phi - phi_init) ** 2)))
    print(f"  -> neg={n_final}  L2={l2:.4f}  ({corr_time:.2f}s)")

    plot_grid_before_after(deformation, phi, title=label)
    plot_neg_jdet_neighborhoods(deformation, phi, title=label)
    return phi

In [ ]:
for variant_name, test_results in all_variant_results.items():
    print(f"\n{'='*70}")
    print(f"  {variant_name}")
    print(f"{'='*70}")
    for i, res in enumerate(test_results):
        label = f"{variant_name} pair {i}"
        to_dvfopt_and_correct(res["deformation"], label)

## Summary: Direct vs Diffeomorphic

In [ ]:
if all_variant_results:
    print(f"{'Variant':<30s}  {'Pair':>5s}  {'Neg Jdet':>10s}  {'Min Jdet':>10s}  {'Inf (ms)':>10s}")
    print("-" * 72)
    for variant_name, test_results in all_variant_results.items():
        for i, res in enumerate(test_results):
            phi = np.stack([res["deformation"][1, 0], res["deformation"][2, 0]])
            jac = jacobian_det2D(phi)
            n_neg = int((jac <= 0).sum())
            min_j = float(jac.min())
            print(f"{variant_name:<30s}  {i:>5d}  {n_neg:>10d}  {min_j:>+10.6f}  "
                  f"{res['inf_time']*1000:>10.2f}")

    # Aggregate
    print()
    for variant_name, test_results in all_variant_results.items():
        total_neg = sum(
            int((jacobian_det2D(np.stack([r["deformation"][1, 0], r["deformation"][2, 0]])) <= 0).sum())
            for r in test_results
        )
        print(f"  {variant_name:<30s}  total neg-Jdet across all pairs: {total_neg}")

## Summary

Key takeaways:
- **Direct regression** (`integration_steps=0`) typically produces more negative Jacobians — the model has no built-in diffeomorphic constraint
- **Diffeomorphic integration** (`integration_steps=7`) uses scaling-and-squaring to compose small velocity fields, producing smoother and more topology-preserving deformations
- **DVFopt correction** can clean up residual folding from either variant, with the diffeomorphic version requiring less correction